In [2]:
#Generic
import os
import time
from datetime import datetime
import argparse
import sys
from glob import glob
#ML libraries
import mlflow
import matplotlib.pyplot as plt
import numpy as np


#Custom
from ptychopinn_torch.utils import load_all_configs_from_mlflow
from ptychopinn_torch.reassembly import reconstruct_image_barycentric
from ptychopinn_torch.config_params import update_existing_config
from ptychopinn_torch.config_params import DataConfig, ModelConfig, TrainingConfig, InferenceConfig, DatagenConfig
from ptychopinn_torch.utils import load_config_from_json, validate_and_process_config, remove_all_files
from ptychopinn_torch.dataloader import PtychoDataset

In [6]:
test = glob(relative_mlflow_path + '/607544705844869811/*')
test[0].split('/')[-1]

'06822d7239504a93ae0f7a6c4577cdc8'

In [3]:
relative_mlflow_path = '/home/beams/HRUTH/code/ptycho_development/CDI-PINN/mlruns'
for file in glob(relative_mlflow_path + '/607544705844869811/*'):
    run_id = file.split('/')[-1]
    print(f"File ID is: {run_id}")
    tracking_uri = f"file:{os.path.abspath(relative_mlflow_path)}"
    mlflow.set_tracking_uri(tracking_uri)
    model_uri = f"runs:/{run_id}/model"

    #Loading config
    data_config, model_config, training_config, inference_config, datagen_config = load_all_configs_from_mlflow(run_id,
                                                                                        tracking_uri)
    
    #Loading model
    model_load_start = time.time()
    loaded_model = mlflow.pytorch.load_model(model_uri)
    loaded_model.to(training_config.device)
    loaded_model.training = True
    model_load_time = time.time() - model_load_start

    print(f"Model loaded successfully at model uri {model_uri}")


File ID is: 06822d7239504a93ae0f7a6c4577cdc8
Model loaded successfully at model uri runs:/06822d7239504a93ae0f7a6c4577cdc8/model
File ID is: 0908cd113f774d15802f41e40b3a51e2
Model loaded successfully at model uri runs:/0908cd113f774d15802f41e40b3a51e2/model
File ID is: 1cda8280703748fabba173f747fc4103
Model loaded successfully at model uri runs:/1cda8280703748fabba173f747fc4103/model
File ID is: 345aa234e8f34935af11c3ebed167448
Model loaded successfully at model uri runs:/345aa234e8f34935af11c3ebed167448/model
File ID is: 3d2ca583357c43baa6ab17519d500355
Model loaded successfully at model uri runs:/3d2ca583357c43baa6ab17519d500355/model
File ID is: 6fb4668f21e44e0b80056f64fdfedf01
Model loaded successfully at model uri runs:/6fb4668f21e44e0b80056f64fdfedf01/model
File ID is: 74ba23396c4042afb1751afe9fa87520
Model loaded successfully at model uri runs:/74ba23396c4042afb1751afe9fa87520/model
File ID is: f637381fd7fe49158bb0ed2e7a28ca45
Model loaded successfully at model uri runs:/f637381

# Moving files to right directory

In [9]:
import os
import shutil
import yaml
from pathlib import Path

In [10]:
experiment_ids = {
    #Dead Leaves Synthetic (Figure 2), Single experiment transfer
    'PS_TP1': 'f637381fd7fe49158bb0ed2e7a28ca45',
    'PS_TP2': '6fb4668f21e44e0b80056f64fdfedf01',
    'PS_IC1': '345aa234e8f34935af11c3ebed167448',
    'PS_IC2': '06822d7239504a93ae0f7a6c4577cdc8',
    'PS_NCM': '0908cd113f774d15802f41e40b3a51e2',
    'PS_FLY1': '3d2ca583357c43baa6ab17519d500355',
    'PS_W': '74ba23396c4042afb1751afe9fa87520',
    'PS_LFP': '1cda8280703748fabba173f747fc4103',

    #Experiment-only (Figure 2), Single experiment transfer
    #To-DO (remove later) copy to data directory
    'PE_TP1': 'c86ba4cc6d424a8fb1370bcfa87d967c',
    'PE_TP2': '3dcc4ce0423c46f6bab529294886f453',
    'PE_IC1': 'b1f8f06f9dee41e48e0323d295e0a5d3',
    'PE_IC2': 'e09fa3d8b48e406aa7c9ff78e34f7782',
    'PE_NCM': '293d4107954d4d11832fe979e6045229',
    'PE_FLY1': '4911939f91d147348f450ec1d78811dd',
    'PE_W': '3360765d399443d0a758e9667c8455b5',
    'PE_LFP': 'aee5ab755d8e4e558c1f328491adb0fb',

    #Multi-model (Figure 3) Single probe/baselines
    #These are the same, you just switch out the evaluation (test) dataset.
    'Single_W': 'bec5909f8c7d407dbdcdc24357495239',
    'Single_LFP': 'e4a5907f060c4af1adb9005a4fc1c51e',
    'Single_FLY1': '9c87bda4384f42a9ad8f7fa0bcc6ea5b',
    'Single_IC2': '5c668cbdb3244e92acd8cb1d682234df',

    #Multi-model (Figure 3) Dual probe
    'W_LFP': 'd3dc54d3897941e89bc16c85b82fba44',
    'W_FLY1': 'cb2e664b3001486ba1047936d0b4a533',
    'W_IC2': '92f5a0518ea641adb8a75e02242c390e',
    'LFP_FLY1': 'f7c72e4058994eea9e9c43ffe7e3b278',
    'LFP_IC2': 'f7c72e4058994eea9e9c43ffe7e3b278',
    'FLY1_IC2': '40f07429768142c5b4d7b57f5c072862',

    #Multi-model (Figure 3) Unified
    'Unified': 'f6ce8d9583164c84955fc6209c340e04',

    #Synthetic objects (Figure 4)
    # Dead leaves
    "DL_W": "74ba23396c4042afb1751afe9fa87520",
    "DL_FLY1": "3d2ca583357c43baa6ab17519d500355",
    "DL_IC2": "6fb4668f21e44e0b80056f64fdfedf01",
    "DL_LFP": "1cda8280703748fabba173f747fc4103",

    # Procedural
    "PR_W": "ed7cabe320b94e82a2a8c13b519abc6d",
    "PR_FLY1": "11ff4f97061f4554b831c45caa40f495",
    "PR_IC2": "04106a3750184a2dbe544cc2c8c2f857",
    "PR_LFP": "1aaf4df1f3a5440da898667953dd11f3",
    
    # Blurred White Noise
    "BWN_W": "42d9fb49f2c94a44b0f03b7f221cab8f",
    "BWN_FLY1": "1c5528183d834c299e2d43e535fb8cf8",
    "BWN_IC2": "0c975e1328e64f7898abb6d94973bbec",
    "BWN_LFP": "0248c08bb3c04004a6672a201f9a3a2d",

    #Simplex
    "SN_W": "7c0b74ecf2d54198acea22aee60fdfc2",
    "SN_FLY1": "db698d0b702348ae8f33bf44e0d1573b",
    "SN_IC2": "c98fefc6252e494db5f0ce5155904333",
    "SN_LFP": "12ce5db7c1564fd7a5fed78704b0c010"

}

In [12]:
def copy_mlflow_experiments(experiment_ids):
    """
    Copy MLFlow experiment directories and modify meta.yaml files
    """
    
    # Define paths
    source_base = Path("/local/CDI-PINN/mlruns/607544705844869811")
    dest_base = Path("/local/CDI-PINN/mlruns_manuscript/607544705844869811")
    
    # Create destination directory if it doesn't exist
    dest_base.mkdir(parents=True, exist_ok=True)
    print(f"Created destination directory: {dest_base}")
    
    # Track processed run_ids to avoid duplicates
    processed_runs = set()
    
    # Process each experiment
    for experiment_name, run_id in experiment_ids.items():
        print(f"\nProcessing {experiment_name}: {run_id}")
        
        # Skip if already processed (handles duplicates in your dictionary)
        if run_id in processed_runs:
            print(f"  Skipping {run_id} - already processed")
            continue
            
        source_dir = source_base / run_id
        dest_dir = dest_base / run_id
        
        # Check if source directory exists
        if not source_dir.exists():
            print(f"  WARNING: Source directory does not exist: {source_dir}")
            continue
            
        # Copy the entire run directory
        if dest_dir.exists():
            print(f"  Destination already exists, removing: {dest_dir}")
            shutil.rmtree(dest_dir)
            
        print(f"  Copying {source_dir} -> {dest_dir}")
        shutil.copytree(source_dir, dest_dir)
        
        # Modify meta.yaml file
        meta_file = dest_dir / "meta.yaml"
        if meta_file.exists():
            try:
                # Read the meta.yaml file
                with open(meta_file, 'r') as f:
                    meta_data = yaml.safe_load(f)
                
                # Clear the artifact_uri
                if 'artifact_uri' in meta_data:
                    original_uri = meta_data['artifact_uri']
                    meta_data['artifact_uri'] = ''
                    print(f"  Modified artifact_uri: '{original_uri}' -> ''")
                
                # Write back the modified meta.yaml
                with open(meta_file, 'w') as f:
                    yaml.safe_dump(meta_data, f, default_flow_style=False)
                    
            except Exception as e:
                print(f"  ERROR modifying meta.yaml: {e}")
        else:
            print(f"  WARNING: meta.yaml not found in {dest_dir}")
            
        processed_runs.add(run_id)
        print(f"  Successfully processed {experiment_name}")
    
    print(f"\nProcessing complete! Processed {len(processed_runs)} unique runs.")
    print(f"Results saved to: {dest_base}")

def verify_copy():
    """
    Verify that the copy was successful by checking a few key files
    """
    dest_base = Path("/local/CDI-PINN/mlruns_manuscript/607544705844869811")
    
    if not dest_base.exists():
        print("Destination directory doesn't exist - copy may have failed")
        return
        
    # Count copied directories
    copied_dirs = [d for d in dest_base.iterdir() if d.is_dir()]
    print(f"\nVerification: Found {len(copied_dirs)} copied run directories")
    
    # Check a sample meta.yaml file
    if copied_dirs:
        sample_dir = copied_dirs[0]
        meta_file = sample_dir / "meta.yaml"
        if meta_file.exists():
            with open(meta_file, 'r') as f:
                meta_data = yaml.safe_load(f)
            print(f"Sample meta.yaml artifact_uri: '{meta_data.get('artifact_uri', 'NOT FOUND')}'")

In [13]:
copy_mlflow_experiments(experiment_ids)

Created destination directory: /local/CDI-PINN/mlruns_manuscript/607544705844869811

Processing PS_TP1: f637381fd7fe49158bb0ed2e7a28ca45
  Copying /local/CDI-PINN/mlruns/607544705844869811/f637381fd7fe49158bb0ed2e7a28ca45 -> /local/CDI-PINN/mlruns_manuscript/607544705844869811/f637381fd7fe49158bb0ed2e7a28ca45
  Modified artifact_uri: 'file:///local/CDI-PINN/mlruns/607544705844869811/f637381fd7fe49158bb0ed2e7a28ca45/artifacts' -> ''
  Successfully processed PS_TP1

Processing PS_TP2: 6fb4668f21e44e0b80056f64fdfedf01
  Copying /local/CDI-PINN/mlruns/607544705844869811/6fb4668f21e44e0b80056f64fdfedf01 -> /local/CDI-PINN/mlruns_manuscript/607544705844869811/6fb4668f21e44e0b80056f64fdfedf01
  Modified artifact_uri: 'file:///local/CDI-PINN/mlruns/607544705844869811/6fb4668f21e44e0b80056f64fdfedf01/artifacts' -> ''
  Successfully processed PS_TP2

Processing PS_IC1: 345aa234e8f34935af11c3ebed167448
  Copying /local/CDI-PINN/mlruns/607544705844869811/345aa234e8f34935af11c3ebed167448 -> /local/